# Two-Channel Spectral Bleed-Through Correction

This notebook demonstrates the core directed two-channel workflow: one source channel bleeds into one target channel, and only the target channel is corrected.

Author: Fabrizio Musacchio  
Date: June 2026

Run the notebook from top to bottom for the packaged example data. To adapt it to your own microscopy data, change the input path and selected channels first, then tune method-specific parameters as described in the Markdown cells.

Several cells open napari viewers. If you run on a headless system, skip those visualization cells and keep the processing and saving cells.


## Imports

Import the package functions used below and locate the repository root. In notebook form, the project root is discovered from the current working directory so the notebook can be opened either from the repository root or from this `notebooks` folder.


In [1]:
from __future__ import annotations 

from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "spectral_unmixing").exists() and (candidate / "example_data").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the spectral-unmixing project root. "
                       "Start this notebook from inside the repository.")

from spectral_unmixing import (
    report_path_from_output_path,
    show_unmixed_channels_in_napari,
    unmix)

In [2]:
# verify that the project root is correct:
print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/husker/Science/Python/Projekte/Spectral Unmixing


## Input And Output Paths

Define the example input stack and all output targets used below.

In fact, you just need to set ``INPUT_PATH`` to your own data and the rest will be 
automatically generated in a subfolder of the input file's parent directory.


In [3]:
# define the input path to the example dataset:
INPUT_PATH = PROJECT_ROOT / "example_data" / "PICASSO_examples" / "2_color_unmixing_validation.tif"
INPUT_NAME = INPUT_PATH.stem
OUTPUT_DIR = INPUT_PATH.parent / "unmixed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# verify that the input path exists:
if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Input path does not exist: {INPUT_PATH}")
else:
    print(f"Input path exists: {INPUT_PATH}")

Input path exists: /Users/husker/Science/Python/Projekte/Spectral Unmixing/example_data/PICASSO_examples/2_color_unmixing_validation.tif


## Optional: Inspect Prepared Stacks In napari

Open the input stack in napari before processing. This is optional but useful for checking channel order, signal quality, and whether the selected example looks as expected.


In [4]:
from spectral_unmixing.viewer import show_all_channels_in_napari
show_all_channels_in_napari(INPUT_PATH, layer_prefix="2-channel example")

Viewer(mouse_move_callbacks=[], mouse_wheel_callbacks=[<function dims_scroll at 0x147d32160>], mouse_drag_callbacks=[<function drag_to_zoom at 0x147d322a0>], mouse_double_click_callbacks=[<function double_click_to_zoom at 0x147d32200>], camera=Camera(center=(0.0, 92.38757249879541, 92.38757249879541), zoom=4.460416379833333, angles=(0.0, 0.0, 0.0), perspective=0.0, mouse_pan=True, mouse_zoom=True, orientation=(<DepthAxisOrientation.TOWARDS: 'towards'>, <VerticalAxisOrientation.DOWN: 'down'>, <HorizontalAxisOrientation.RIGHT: 'right'>)), cursor=Cursor(position=(0.0, 0.0, 0.0, 0.0), viewbox=None, scaled=True, size=1.0, style=<CursorStyle.STANDARD: 'standard'>), dims=Dims(ndim=4, ndisplay=2, order=(0, 1, 2, 3), axis_labels=('-4', '-3', '-2', '-1'), rollable=(True, True, True, True), range=(RangeTuple(start=0.0, stop=0.0, step=1.0), RangeTuple(start=0.0, stop=0.0, step=1.0), RangeTuple(start=0.0, stop=184.77514499759081, step=0.3084726961562451), RangeTuple(start=0.0, stop=184.775144997590

## Fixed Alpha Example

Run unmixing with a manually chosen fixed bleed-through coefficient.

Method summary:

- ``alpha_mode="fixed"`` means no alpha is estimated from the data.
- ``method="manual"`` documents that the user provides ``alpha`` directly.
- The same alpha is applied to every time point and every z-slice.

What can be adjusted:

- ``alpha``:
  Strength of the source-channel subtraction from the target channel.
  Larger values remove more source contribution from the target channel.
- ``source_channel`` and ``target_channel``:
  Select which channel is treated as the bleeding source and which one is
  corrected.
- ``clip_negative`` inside ``unmix(...)``:
  If enabled, negative corrected values are clipped to zero.

When this is useful:

- Best choice when alpha was measured independently from a proper control.
- Most reproducible and scientifically preferred if acquisition settings are
- stable across experiments.


In [ ]:
# define the output path for the fixed-alpha unmixing result:
OUTPUT_FIXED = OUTPUT_DIR / f"{INPUT_NAME}_unmixed_fixed_alpha.tif"

fixed_output = unmix(
    input_path=INPUT_PATH,
    output_path=OUTPUT_FIXED,
    # source_channel=0,  # default: 0
    # target_channel=1,  # default: 1
    method="manual",
    alpha=0.62,
    # alpha_mode="fixed",  # only relevant for multi-time-point stacks; default: None; other options: "reference_t", "per_t"
    # clip_negative=True,  # default: True
    # output_dtype="float32",  # default: "float32"
    # verbose=True,  # default: True
)

show_unmixed_channels_in_napari(
    fixed_output,
    source_channel=0,
    target_channel=1,
    layer_prefix="Fixed alpha",
    source_colormap="cyan",
    target_colormap="yellow")


## Mean-Ratio Example

Estimate one alpha from an optional reference time point using the ``mean_ratio`` rule.

Method summary:

- ``method="mean_ratio"`` computes alpha as the mean target intensity divided
  by the mean source intensity within bright source voxels.
- All z-slices of the chosen reference time point contribute to that estimate.

What can be adjusted:

- ``alpha_mode="reference_t"`` can be set in case of, e.g., multi-time point stacks. 
  This mode estimates one coefficient per direction from a chosen reference time point 
  and then applies both coefficients to the whole stack.
- ``alpha_reference_t``:
  Reference time point used for both directions. Only relevant for multi-time-point stacks; 
  default: 0.
- ``signal_percentile``:
  Defines how bright source voxels must be to enter the estimation mask.
  Higher values focus more strongly on the brightest source signal.
- ``background_percentile``:
  Controls the low-percentile background estimate subtracted before alpha
  estimation.
- ``target_low_percentile``:
  Optional extra restriction to prefer voxels with low target intensity, which
  can reduce bias from true biological target signal.

Effect of these settings:

- A higher ``signal_percentile`` usually makes alpha estimation more selective
  but also reduces the number of voxels used.
- A different ``alpha_reference_t`` matters when bleed-through or biology
  changes over time (in multi-time-point stacks).


In [ ]:
# define the output path for the reference-time-point unmixing result:
OUTPUT_REFERENCE = OUTPUT_DIR / f"{INPUT_NAME}_unmixed_reference_t0_mean_ratio.tif"

reference_output = unmix(
    input_path=INPUT_PATH,
    output_path=OUTPUT_REFERENCE,
    # source_channel=0,  # default: 0
    # target_channel=1,  # default: 1
    method="mean_ratio",
    #alpha_mode="reference_t",
    #alpha_reference_t=0,
    signal_percentile=99.0,
    background_percentile=1.0,
    # target_low_percentile=95.0,
    # preprocess_alpha_inputs=True,  # default: True
    # clip_negative=True,  # default: True
)
print(reference_output)
print(report_path_from_output_path(reference_output).read_text(encoding="utf-8"))
show_unmixed_channels_in_napari(
    reference_output,
    source_channel=0,
    target_channel=1,
    layer_prefix="mean_ratio")


## Linear-Fit Example

Estimate one alpha from an optional reference time point via masked least squares.

Method summary:

- ``method="linear_fit"`` fits the model ``target ≈ alpha * source`` inside
  the selected voxel mask.
- No intercept is fitted; background is handled by the percentile-based
  preprocessing used for alpha estimation.
- The resulting alpha is then applied to all time points and z-slices.

What can be adjusted:

- ``signal_percentile``:
  Controls which bright source voxels define the fitting mask.
- ``background_percentile``:
  Influences the background-subtracted data used before fitting.
- ``alpha_reference_t``:
  Selects the time point from which the fit is derived.

Effect of these settings:

- Compared with ``mean_ratio``, ``linear_fit`` is often a bit closer to a true
  least-squares estimate and can behave differently when masked intensities
  have broad dynamic ranges.


In [ ]:
# define the output path for the reference-time-point linear-fit unmixing result:
OUTPUT_REFERENCE_LINEAR_FIT = OUTPUT_DIR / f"{INPUT_NAME}_unmixed_reference_t0_linear_fit.tif"

reference_linear_fit_output = unmix(
    input_path=INPUT_PATH,
    output_path=OUTPUT_REFERENCE_LINEAR_FIT,
    # source_channel=0,  # default: 0
    # target_channel=1,  # default: 1
    # alpha_mode="reference_t",
    # alpha_reference_t=0,
    method="linear_fit",
    signal_percentile=99.0,
    background_percentile=1.0,
    # target_low_percentile=95.0,
    # preprocess_alpha_inputs=True,  # default: True
    # clip_negative=True,  # default: True
)
print(reference_linear_fit_output)
print(report_path_from_output_path(reference_linear_fit_output).read_text(encoding="utf-8"))
show_unmixed_channels_in_napari(
    reference_linear_fit_output,
    source_channel=0,
    target_channel=1,
    layer_prefix="Reference linear_fit")


## Corr-Min Example

Estimate alpha by minimizing residual correlation after correction.

Method summary:

- ``method="corr_min"`` searches for the alpha that minimizes the Pearson
  correlation between the source channel and the corrected target channel.
- Intuition: after ideal bleed-through removal, source structure should be less
  visible inside the corrected target channel.

What can be adjusted:

- ``alpha_max``:
  Upper search bound for alpha. Increase it if stronger bleed-through is
  plausible; keep it conservative to avoid overly aggressive subtraction.
- ``signal_percentile`` and ``background_percentile``:
  Still control the source mask and preprocessing used for the optimization.

Effect of these settings:

- ``corr_min`` can be more aggressive than ``mean_ratio`` because it explicitly
  tries to remove statistical dependence.
- If source and target channels are biologically correlated, this method may
  subtract true target signal together with bleed-through.


In [ ]:
# define the output path for the reference-time-point corr-min unmixing result:
OUTPUT_REFERENCE_CORR_MIN = OUTPUT_DIR / f"{INPUT_NAME}_unmixed_reference_t0_corr_min.tif"

reference_corr_min_output = unmix(
    input_path=INPUT_PATH,
    output_path=OUTPUT_REFERENCE_CORR_MIN,
    # source_channel=0,  # default: 0
    # target_channel=1,  # default: 1
    # alpha_mode="reference_t",
    # alpha_reference_t=0,
    method="corr_min",
    signal_percentile=99.0,
    background_percentile=1.0,
    alpha_max=1.0,
    # target_low_percentile=95.0,
    # preprocess_alpha_inputs=True,  # default: True
    # max_alpha_voxels=500_000,  # default
    # random_state=0,  # default
)
print(reference_corr_min_output)
print(report_path_from_output_path(reference_corr_min_output).read_text(encoding="utf-8"))
show_unmixed_channels_in_napari(
    reference_corr_min_output,
    source_channel=0,
    target_channel=1,
    layer_prefix="Reference corr_min")


## MI-Min Example

Estimate forward coefficient by minimizing mutual information.

Method summary:

- ``method="mi_min"`` uses a PICASSO-like two-channel criterion for each
  direction independently.
- ``mi_bins`` controls the histogram-based mutual-information estimate for the
  forward direction.

What can be adjusted:

- ``mi_bins``:
  Histogram resolution used by the mutual-information estimate.
- ``alpha_max``:
  Search bound for the forward optimization.


In [ ]:
# define the output path for the reference-time-point mi-min unmixing result:
OUTPUT_REFERENCE_MI_MIN = OUTPUT_DIR / f"{INPUT_NAME}_unmixed_reference_t0_mi_min.tif"

reference_mi_min_output = unmix(
    input_path=INPUT_PATH,
    output_path=OUTPUT_REFERENCE_MI_MIN,
    # source_channel=0,  # default: 0
    # target_channel=1,  # default: 1
    # alpha_mode="reference_t",
    # alpha_reference_t=0,
    method="mi_min",
    signal_percentile=50.0,
    background_percentile=1.0,
    alpha_max=1.0,
    mi_bins=64,
    # target_low_percentile=95.0,
    # preprocess_alpha_inputs=True,  # default: True
    # max_alpha_voxels=500_000,  # default
    # random_state=0,  # default
)
print(reference_mi_min_output)
print(report_path_from_output_path(reference_mi_min_output).read_text(encoding="utf-8"))
show_unmixed_channels_in_napari(
    reference_mi_min_output,
    source_channel=0,
    target_channel=1,
    layer_prefix="Reference mi_min")


## End

The notebook is complete. Saved outputs can be reopened from the output folder or reused in downstream analysis scripts.
